# 01 — First GPU Slurm job: Aer on an A100/H200

**Level:** beginner → intermediate. **Environment:** CPU Jupyter workbench + approved Aer-GPU Apptainer SIF. **GPU:** one scheduler-allocated device. **Credentials:** your own SSH private key / real POSIX identity, never shared root/sudo credentials. **Expected time:** queue-dependent, requested walltime 02:00:00. **Artifacts:** `results/aer-gpu-JOBID.json` on the submit filesystem, scheduler logs/accounting.

**Not an interactive GPU session.** From Kubernetes, this notebook executes **native `sbatch`, `squeue` and `sacct` on the Slurm login host through SSH authenticated as the *actual researcher***. No browser SSH terminal, no privileged `runuser` helper and no extra SDK. A direct local Slurm client is also supported *if* running as the provisioned real researcher on an authorized submit host.

Prerequisites for ACE: your private research key lives in your own 0700 home (`~/.ssh/`, key mode 0600); its public half is **approved and installed for your own account** on the Slurm login host, host key preverified and saved to known_hosts, and the repo source is accessible at the same `/home/research/<user>/...` NFS path on that host. **P0 SSH, GPU partition, resource quota and SIF image must be operator-approved before submission.** No GPU is reserved while editing this notebook.

For remote Axis H200, submit-side source and results need explicit transfer. The ACE NFS path is not presumed to exist at Axis; this notebook intentionally does not pretend to stage them there. See `docs/M5-BATCH-FIRST-SIMULATOR-PATH.md`.

In [ ]:
import os
import re
import shlex
import shutil
import subprocess
from pathlib import Path

ROOT = next(
    (d for d in (Path.cwd(), *Path.cwd().parents) if (d / "slurm/aer-gpu.sbatch").is_file()),
    None,
)
assert ROOT is not None, "Open from a checkout containing slurm/aer-gpu.sbatch"

RESEARCH_USER = os.getenv("QUANTUM_PLATFORM_USERNAME") or os.getenv("USER")
assert RESEARCH_USER and re.fullmatch(r"[a-z_][a-z0-9_-]*", RESEARCH_USER)
assert os.geteuid() >= 20000, "Expected a provisioned non-service POSIX researcher identity"
expected_uid = os.getenv("QUANTUM_PLATFORM_UID")
assert not expected_uid or os.geteuid() == int(expected_uid), "Workbench UID mismatch"
assert RESEARCH_USER != "root"

PARTITION = os.getenv("SLURM_GPU_PARTITION")
IMAGE = os.getenv("QISKIT_AER_GPU_IMAGE")
SITE = os.getenv("SLURM_EXECUTION_SITE", "ace-a100")
assert PARTITION and re.fullmatch(r"[A-Za-z][A-Za-z0-9_.-]*", PARTITION)
assert IMAGE and Path(IMAGE).is_absolute() and "," not in IMAGE and "\n" not in IMAGE
assert re.fullmatch(r"[a-zA-Z0-9_.-]+", SITE)

SLURM_LOGIN_HOST = os.getenv("SLURM_LOGIN_HOST", "").strip()
SSH = None
if SLURM_LOGIN_HOST:
    assert re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9.-]*", SLURM_LOGIN_HOST)
    assert shutil.which("ssh"), "Workbench is missing the OpenSSH client"
    key = Path(os.getenv("SLURM_RESEARCHER_SSH_KEY", "~/.ssh/id_ed25519")).expanduser()
    known_hosts = Path(os.getenv("SLURM_RESEARCHER_KNOWN_HOSTS", "~/.ssh/known_hosts")).expanduser()
    assert key.is_file() and key.stat().st_uid == os.geteuid()
    assert (key.stat().st_mode & 0o077) == 0, "Private SSH key must have mode 0600"
    assert known_hosts.is_file() and known_hosts.stat().st_size > 0
    assert str(ROOT.resolve()).startswith(f"/home/research/{RESEARCH_USER}/"), (
        "ACE submission needs code at the researcher's site-visible shared NFS path"
    )
    SSH = [
        "ssh", "-n", "-i", str(key),
        "-o", "BatchMode=yes", "-o", "IdentitiesOnly=yes",
        "-o", "StrictHostKeyChecking=yes",
        "-o", f"UserKnownHostsFile={known_hosts}",
        "-o", "ConnectTimeout=10",
        f"{RESEARCH_USER}@{SLURM_LOGIN_HOST}",
    ]
else:
    assert shutil.which("sbatch"), (
        "Set SLURM_LOGIN_HOST and your own researcher SSH identity from the Jupyter workbench, "
        "or run this on an authorized Slurm login host with native sbatch"
    )

def slurm_command(args, *, check=True):
    # SSH authenticates directly as the provisioned researcher; no sudo/runuser helper.
    command = SSH + [shlex.join(args)] if SSH else args
    return subprocess.run(
        command, cwd=ROOT, text=True, capture_output=True, check=check, timeout=30
    )

print("Research user:", RESEARCH_USER, "UID:", os.geteuid(), "site:", SITE)
print("Transport:", "per-researcher SSH → native Slurm" if SSH else "local native Slurm")
print("Approved GPU partition:", PARTITION, "approved SIF:", IMAGE)

## Submit once (one GPU)

This is an explicit user-triggered action. Re-running this cell submits **another** job. Do not retry blindly after a network failure: inspect your queue first. Never run as the portal's service account.

In [ ]:
# CAUTION: executing this cell again submits another job.
command = [
    "sbatch", "--parsable", f"--partition={PARTITION}",
    f"--export=QISKIT_AER_GPU_IMAGE={IMAGE},AER_QUBITS=22,AER_SHOTS=2048",
    str(ROOT / "slurm/aer-gpu.sbatch"),
]
response = slurm_command(command)
JOB_ID = response.stdout.strip().split(";", 1)[0]
assert JOB_ID.isdigit(), f"Unexpected Slurm job ID: {response.stdout!r}"

import json
STATE_DIR = Path.home() / ".local" / "state" / "quantum-workflows"
STATE_DIR.mkdir(parents=True, exist_ok=True, mode=0o700)
STATE_FILE = STATE_DIR / f"aer-gpu-{SITE}-{JOB_ID}.json"
STATE_FILE.write_text(json.dumps({
    "job_id": JOB_ID, "site": SITE, "partition": PARTITION, "researcher": RESEARCH_USER,
}, indent=2) + "\n")
STATE_FILE.chmod(0o600)
print(f"Submitted durable Slurm job {JOB_ID}; notebook may close now.")
print("Saved private job reference:", STATE_FILE)

## Observe without reserving the GPU yourself

The Slurm job survives notebook/browser closure. On restart, restore your own job ID after authenticating; do not scan other users' records.

In [ ]:
import json
from pathlib import Path
STATE_DIR = Path.home() / ".local" / "state" / "quantum-workflows"
if "JOB_ID" not in globals():
    saved_jobs = sorted(
        STATE_DIR.glob(f"aer-gpu-{SITE}-*.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    ) if STATE_DIR.exists() else []
    assert saved_jobs, "No recorded job. Restore your own job ID after running setup cell."
    state = json.loads(saved_jobs[0].read_text())
    assert state["site"] == SITE and state["researcher"] == RESEARCH_USER
    JOB_ID = str(state["job_id"])
    print("Restored job", JOB_ID, "on site", SITE)
assert str(JOB_ID).isdigit()
for args in (
    ["squeue", "-h", "-j", str(JOB_ID), "-o", "%i|%u|%T|%M|%R"],
    ["sacct", "-nP", "-X", "-j", str(JOB_ID),
     "--format=JobID,User,State,ExitCode,Elapsed,AllocTRES"],
):
    proc = slurm_command(args, check=False)
    print(args[0], proc.stdout.strip() or proc.stderr.strip())

## Verify scientific output, not just a COMPLETED state

Only local/shared-filesystem mode is implemented by this reference lab. The H200 connector must explicitly stage outputs back and validate checksum, ownership and manifest; it must not silently attempt a cross-site NFS or SSHFS mount.

In [ ]:
import json
assert "JOB_ID" in globals()
if "ROOT" not in globals():
    ROOT = next((d for d in (Path.cwd(), *Path.cwd().parents)
                 if (d / "slurm/aer-gpu.sbatch").is_file()), None)
assert ROOT is not None, "Open the notebook from the source checkout to find local results"
output = ROOT / "results" / f"aer-gpu-{JOB_ID}.json"
if output.is_file():
    data = json.loads(output.read_text())
    assert data["scientific_check_passed"], "Scheduler completed but numerical validation failed"
    print({key: data.get(key) for key in ("slurm_job_id","qubits","shots","duration_seconds","qiskit_aer_version","scientific_check_passed")})
else:
    print("Not yet staged:", output, "Check sacct and authorized site stage-out.")

**Cleanup:** inspect your output and remove only disposable files according to facility retention policy. To cancel **your own** running job, use native `scancel JOB_ID` under your own Slurm identity (via `slurm_command(["scancel", str(JOB_ID)])` if required). The notebook never cancels automatically.

This is a P0/P2 reference: real `nlisa` acceptance and GPU isolation require site-operator evidence. Native SSH authentication alone does not prove the Slurm GPU partition is configured.